# **Staff Data Cleaning** 

## 1. Importing Libraries and Loading Data

In [79]:
import pandas as pd
import numpy as np

from pathlib import Path

import sys
sys.path.append(str(Path.cwd().parent))

from scripts.paths import RAW_DIR, CLEANED_DIR

df = pd.read_csv(RAW_DIR/"03_staff_dirty.csv")

## 2. Initial Data Inspection

In [80]:
df.head()

,Staff_ID,Staff_Name,Gender,Role,Department_ID,Joining_Date,Shift,Monthly_Salary,Experience_Years
0,S10000,Amit Chouhan,F,Pharmacist,D017,2026-04-13,Evening,137682,22
1,S10001,Rohit Mehta,F,Attendant,D004,2017-05-10,Night,172854,11
2,S10002,Dr Aarav Chouhan,Female,Pharmacist,D007,2019-02-11,Night,106093,22
3,S10003,Ananya Gour,F,Admin,D017,2026-05-16,morning,124005,25
4,S10004,Nisha Joshi,M,Technician,D005,2018-07-09,Night,-12000,23


In [81]:
df.shape

(900, 9)

In [82]:
df.columns.tolist()

['Staff_ID',
 'Staff_Name',
 'Gender',
 'Role',
 'Department_ID',
 'Joining_Date',
 'Shift',
 'Monthly_Salary',
 'Experience_Years']

In [83]:
df.dtypes

Staff_ID            object
Staff_Name          object
Gender              object
Role                object
Department_ID       object
Joining_Date        object
Shift               object
Monthly_Salary      object
Experience_Years     int64
dtype: object

In [84]:
df.isna().sum()

Staff_ID              0
Staff_Name            0
Gender              175
Role                  0
Department_ID        20
Joining_Date          0
Shift               131
Monthly_Salary       11
Experience_Years      0
dtype: int64

In [85]:
df.duplicated().sum()

0

## 3. Audit of Specific Columns

In [86]:
df["Staff_ID"].duplicated().sum()

0

In [87]:
df["Gender"].unique()

array(['F', 'Female', 'M', 'Male', nan], dtype=object)

In [88]:
df["Role"].unique()

array(['Pharmacist', 'Attendant', 'Admin', 'Technician', 'Nurse',
       'Doctor', 'Receptionist'], dtype=object)

In [89]:
df["Department_ID"].unique()

array(['D017', 'D004', 'D007', 'D005', 'D008', 'D014', 'D016', 'D013',
       'D999', 'D015', 'D011', 'D012', 'D006', 'D001', 'D002', 'D018',
       'D010', 'D003', 'D009', nan], dtype=object)

Below we've done a referential-integrity check, in our last file "02_departments.csv" and in this file we have "Department_ID" column in common

In [90]:
departments_df = pd.read_csv(CLEANED_DIR/"02_departments_cleaned.csv")

In [91]:
valid_department_ids = departments_df["Department_ID"].dropna().unique() 

In [92]:
df.loc[~df["Department_ID"].isin(valid_department_ids), "Department_ID"]

11     D999
174     NaN
189     NaN
205     NaN
216     NaN
272     NaN
274     NaN
409     NaN
412    D999
421    D999
443     NaN
521     NaN
535     NaN
540    D999
576    D999
591     NaN
595     NaN
603    D999
619     NaN
632     NaN
636    D999
665     NaN
667     NaN
692     NaN
704     NaN
725    D999
786     NaN
813    D999
874     NaN
883    D999
Name: Department_ID, dtype: object

In [93]:
df["Department_ID"].value_counts(dropna = False)

Department_ID
D001    62
D008    62
D015    56
D014    54
D006    52
D003    50
D007    48
D016    48
D018    48
D017    46
D005    45
D004    45
D010    44
D011    44
D013    44
D012    43
D002    42
D009    37
NaN     20
D999    10
Name: count, dtype: int64

In [94]:
df["Shift"].unique()

array(['Evening', 'Night ', 'Night', 'morning', nan, 'Morning'],
      dtype=object)

In [95]:
df["Joining_Date"].dtype

dtype('O')

In [96]:
df["Monthly_Salary"].describe()

count       889
unique      865
top       45000
freq         10
Name: Monthly_Salary, dtype: object

In [97]:
df["Monthly_Salary"].unique()

array(['137682', '172854', '106093', '124005', '-12000', '153987',
       '104383', '165059', '132720', '122005', '132517', '50213', '58522',
       '39024', '136090', '146257', '92150', '85689', '99421', '46406',
       '108580', '116984', '112360', '135895', '92964', '74187', '144161',
       '106054', '175288', '156390', '66154', '157703', '37576', '74534',
       '56130', '159215', '48526', '150576', '21703', '42856', '65430',
       '179464', '113962', '110212', '149601', '89500', '90004', '101063',
       '135293', '45,000', '103127', '61521', '120374', '87103', nan,
       '135229', '46337', '97849', '167670', '39968', '69808', '46211',
       '28881', '87388', '131083', '115419', '55513', '148636', '99549',
       '32517', '86759', '111277', '131194', '83888', '102139', '91353',
       '73327', '35618', '64610', '168729', '45000', '80347', '156449',
       '64265', '128703', '101649', '44001', '58131', '171828', '84916',
       '136198', '66628', '168189', '170633', '52443', '1

In [98]:
df["Monthly_Salary"].value_counts(dropna = False)

Monthly_Salary
NaN       11
45000     10
45,000     8
-12000     6
39968      2
          ..
41231      1
38616      1
57665      1
117576     1
72692      1
Name: count, Length: 866, dtype: int64

In [99]:
salary_temp = df["Monthly_Salary"].replace(",", "", regex = False)

In [100]:
salary_numeric = pd.to_numeric(salary_temp, errors = "coerce")

In [101]:
salary_numeric.isna().sum()

19

In [102]:
(salary_numeric < 0).sum()

6

In [103]:
salary_numeric.describe()

count       881.000000
mean      98550.553916
std       46161.936123
min      -12000.000000
25%       58522.000000
50%       98823.000000
75%      138203.000000
max      179987.000000
Name: Monthly_Salary, dtype: float64

In [104]:
df.loc[salary_numeric < 0, "Monthly_Salary"]

4      -12000
31     -12000
97     -12000
651    -12000
706    -12000
759    -12000
Name: Monthly_Salary, dtype: object

In [105]:
df["Experience_Years"].describe()

count    900.000000
mean      13.894444
std        8.539314
min        0.000000
25%        7.000000
50%       14.000000
75%       22.000000
max       28.000000
Name: Experience_Years, dtype: float64

In [106]:
df["Staff_Name"].str.istitle().value_counts(dropna = False)

Staff_Name
True    900
Name: count, dtype: int64

## 4. Data Cleaning

### 4.1 Department_ID

In [107]:
department_cleaned = df["Department_ID"].replace("D999", np.nan)

In [108]:
department_cleaned.isna().sum()

30

In [109]:
department_cleaned.unique()

array(['D017', 'D004', 'D007', 'D005', 'D008', 'D014', 'D016', 'D013',
       nan, 'D015', 'D011', 'D012', 'D006', 'D001', 'D002', 'D018',
       'D010', 'D003', 'D009'], dtype=object)

In [110]:
df["Department_ID"] = department_cleaned

### 4.2 Gender

In [111]:
Gender_cleaned = df["Gender"].replace(
    {
        "M": "Male",
        "F": "Female"
    }
)

In [112]:
Gender_cleaned.unique()

array(['Female', 'Male', nan], dtype=object)

In [113]:
Gender_cleaned.isna().sum()

175

In [114]:
df["Gender"] = Gender_cleaned

### 4.3 Monthly_Salary

In [115]:
salary_cleaned = salary_numeric.mask(salary_numeric < 0)

In [116]:
salary_cleaned.isna().sum()

25

In [117]:
salary_cleaned.dtype

dtype('float64')

In [118]:
salary_cleaned = salary_cleaned.astype("Int64")

In [119]:
salary_cleaned.dtype

Int64Dtype()

In [120]:
df["Monthly_Salary"] = salary_cleaned

### 4.4 Shift

In [121]:
shift_cleaned = df["Shift"].str.strip().str.title()

In [122]:
shift_cleaned.unique()

array(['Evening', 'Night', 'Morning', nan], dtype=object)

In [123]:
df["Shift"] = shift_cleaned

### 4.5 Joining Date

In [124]:
joining_cleaned = pd.to_datetime(df["Joining_Date"], errors = "coerce")

In [125]:
joining_cleaned.dtype

dtype('<M8[ns]')

In [126]:
joining_cleaned.isna().sum()

0

In [127]:
df["Joining_Date"] = joining_cleaned

## 5. Final Validation

In [128]:
df.shape

(900, 9)

In [129]:
df.dtypes

Staff_ID                    object
Staff_Name                  object
Gender                      object
Role                        object
Department_ID               object
Joining_Date        datetime64[ns]
Shift                       object
Monthly_Salary               Int64
Experience_Years             int64
dtype: object

In [130]:
df.isna().sum()

Staff_ID              0
Staff_Name            0
Gender              175
Role                  0
Department_ID        30
Joining_Date          0
Shift               131
Monthly_Salary       25
Experience_Years      0
dtype: int64

In [131]:
df.duplicated().sum()

0

In [132]:
df["Staff_ID"].duplicated().sum()

0

In [133]:
df["Gender"].unique()

array(['Female', 'Male', nan], dtype=object)

In [134]:
df["Shift"].unique()

array(['Evening', 'Night', 'Morning', nan], dtype=object)

In [135]:
df["Monthly_Salary"].min()

18112

## 6. Export Cleaned Data

In [136]:
df.to_csv(CLEANED_DIR/"03_staff_cleaned.csv", index = False)